In [17]:
from timescale import TimescaleDBManager
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd
import os
import json

In [18]:

# import importlib
# import timescale
# importlib.reload(timescale)

# # RE-IMPORTA la clase para que el nombre apunte a la nueva versión
# from timescale import TimescaleDBManager 

# Ahora tu instancia debería funcionar correctamente

In [19]:
tabla_origen  = 'metricas_logs'
tabla_destino = 'vectores_logs'

In [20]:
def obtener_datos_ejecucion(execution_name, engine,estandarizar=True):
    """
    Extrae todos los registros de una ejecución y métrica específica.
    """
    if estandarizar:
        SQL = f"""
            SELECT
                m.time,m.metric_name,m.instance,m.grupo,m.job,m.tags,m.label,
                (m.metric_value - e.media) / NULLIF(e.desv_std, 0) AS metric_value
            FROM {tabla_origen} m
            INNER JOIN t_escalado e
                ON e.tabla = '{tabla_origen}'
               AND e.execution_name = m.execution_name
               AND e.metric_name = m.metric_name
            
            WHERE m.execution_name = %s 
            ORDER BY time ASC, metric_name ASC, instance ASC, grupo ASC,job ASC

        """
    else:
        SQL = f"""
            SELECT time,metric_name,instance,grupo,job,tags,label,metric_value
            FROM {tabla_origen}
            WHERE execution_name = %s 
            ORDER BY time ASC, metric_name ASC, instance ASC, grupo ASC,job ASC
            
        """
    
    # Ejecutamos la consulta pasando los parámetros de forma segura
    df = pd.read_sql(SQL, engine, params=(execution_name,))
    
    # Convertimos la columna time a formato datetime
    df['time'] = pd.to_datetime(df['time'])
    df         = df.sort_values('time')

    inicio_global = df['time'].min()
    df['tiempo_relativo'] = (df['time'] - inicio_global).dt.total_seconds()
        
    return df

In [21]:
def procesar_datos(lista_de_datos,clave_label,clave_vector):
    lista_result=[]
    if lista_de_datos:
        set_valores = {d[clave_label] for d in lista_de_datos}
        if len(set_valores) == 1:
            label_iguales=True
            valor=set_valores.pop()
            for item in lista_de_datos:
                lista_result.extend(item[clave_vector])
        else:
            #hay valores distintos
            label_iguales=False
            valor=None
    else:
        label_iguales=False
        valor=None
    return label_iguales,valor,lista_result

In [22]:
load_dotenv()
user_env        = os.getenv("USER_DB")
pass_env        = os.getenv("PASS_DB")
host_env        = os.getenv("HOST_DB", "localhost")
port_env        = os.getenv("DB_PORT_EXPOSED", "5432")
db_env          = os.getenv("NAME_DB", "tfm_db")
FOLDER_SALIDA   = os.getenv("FOLDER_SALIDA")
FOLDER_TMP      = os.getenv("FOLDER_TMP")

SEED            = os.getenv("SEED")

db = TimescaleDBManager(
    user=user_env,
    password=pass_env,
    host=host_env,
    port=port_env,
    dbname=db_env
)
db.conectar()
conexion_postgresql = db.connection

#esto es para usar una conexion postgrews para pandas que da un warning indicando que solo esta testeado para sqlalchemy
engine = create_engine(f'postgresql+psycopg2://{user_env}:{pass_env}@{host_env}:{port_env}/{db_env}')

[OK] Conexión establecida con TimescaleDB.


In [23]:
df_ejecuciones_outlier = pd.read_parquet(f'{FOLDER_SALIDA}/df_ejecuciones_outlier(metricas_logs).parquet')
df_outliers_pca = pd.read_parquet(f'{FOLDER_SALIDA}/df_outliers_pca(metricas_logs).parquet')

In [24]:
lista_ejecuciones_outlier = df_ejecuciones_outlier['execution_name'].tolist()

In [25]:
SQL=f'select distinct(execution_name) from {tabla_origen}'
df_execution_name = pd.read_sql(SQL, engine)
lista_execution_name=df_execution_name['execution_name'].to_list()

In [26]:
len(lista_execution_name)

850

In [27]:
SQL=f'select distinct(metric_name) from {tabla_origen}'
df_metric_name = pd.read_sql(SQL, engine)
lista_metric_name_ord=sorted(df_metric_name['metric_name'].to_list())


In [28]:
ruta_archivo = f'{FOLDER_SALIDA}/lista_metricas_filtradas(metricas_logs).json'
with open(ruta_archivo, "r", encoding="utf-8") as f:
    lista_metric_name_ord = json.load(f)
    
#nos aseguradmos que esta ordenada y la guardamos con nombre ord
lista_metric_name_ord.sort()
ruta_archivo = f'{FOLDER_SALIDA}/lista_metricas_filtradas_ord(metricas_logs).json'
with open(ruta_archivo, "w", encoding="utf-8") as f:
    json.dump(lista_metric_name_ord, f, ensure_ascii=False, indent=4)


In [29]:
lista_metric_name_vector=lista_metric_name_ord.copy()

filename=f'{FOLDER_SALIDA}/lista_metric_name_vector_ord(metricas_logs).json'
with open(filename, "w", encoding="utf-8") as f:
    json.dump(lista_metric_name_vector, f, ensure_ascii=False, indent=4)

In [30]:
len(lista_execution_name)

850

In [31]:
len(lista_metric_name_vector)

331

In [32]:
#creacion de vectores con 3 puntos de datos
N=0
for una_ejecucion in lista_execution_name:
    if una_ejecucion in lista_ejecuciones_outlier:
        continue
    df = obtener_datos_ejecucion(una_ejecucion, engine)
    print(f'datos obtenidos execution_name={una_ejecucion}')
    N+=1

    lista_tiempo_relativo=df['tiempo_relativo'].unique().tolist()
  
    lista_reg=[]
    while len(lista_tiempo_relativo)>=3:
        #print('==============')
        lista_t=lista_tiempo_relativo[0:3]
        lista_data=[]
        #print(f't_rel = {lista_t[0]}')
        for t_relativo in lista_t:
            data={}
            #df_t = df[df['tiempo_relativo']==t_relativo]
            df_t = df[df['tiempo_relativo']==t_relativo].sort_values(by=['metric_name', 'instance', 'grupo','job'])
            lista_label=df_t['label'].unique().tolist()
            
            if len(lista_label)>1: #esto no deberia pasar, todos los label de este df deberian ser iguales
                break
            label         = lista_label[0]
            data['label'] = label

            vector = df_t.set_index('metric_name')['metric_value'].reindex(lista_metric_name_vector).tolist()
            #vector         = df_t['metric_value'].to_list()
            data['vector'] = vector
            lista_data.append(data)
            
        valores_iguales,label,vector_unificado=procesar_datos(lista_data,'label','vector')
        #print(f'valores_iguales = {valores_iguales}    label = {label}  ')
        if valores_iguales:
            reg={}
            reg['execution_name'] = una_ejecucion
            reg['t_rel']          = lista_t[0]
            reg['label']          = label
            reg['vector']         = vector_unificado

            #almacenar reg
            lista_reg.append(reg)
        lista_tiempo_relativo.pop(0)
        
    db.insertar_vectores_batch(lista_reg,tabla_destino)
    print(f"execution_name={una_ejecucion}   | N = {N:4d}  |len(vector) = {len(reg['vector'])}",end='\r')
    # if N>0:
    #     break

datos obtenidos execution_name=light-oauth2-data-1719592986.tar
datos obtenidos execution_name=light-oauth2-data-1719594842.tarlen(vector) = 993
datos obtenidos execution_name=light-oauth2-data-1719598443.tarlen(vector) = 993
datos obtenidos execution_name=light-oauth2-data-1719602043.tarlen(vector) = 993
datos obtenidos execution_name=light-oauth2-data-1719605644.tarlen(vector) = 993
datos obtenidos execution_name=light-oauth2-data-1719609252.tarlen(vector) = 993
datos obtenidos execution_name=light-oauth2-data-1719612853.tarlen(vector) = 993
datos obtenidos execution_name=light-oauth2-data-1719616455.tarlen(vector) = 993
datos obtenidos execution_name=light-oauth2-data-1719620054.tarlen(vector) = 993
datos obtenidos execution_name=light-oauth2-data-1719623651.tarlen(vector) = 993
datos obtenidos execution_name=light-oauth2-data-1719627249.tarlen(vector) = 993
datos obtenidos execution_name=light-oauth2-data-1719630850.tarlen(vector) = 993
datos obtenidos execution_name=light-oauth2-d

In [84]:
for item in lista_reg:
    print(f"{item['execution_name']:30s}  {item['label']:50s} {int(item['t_rel']):3d}  {len(item['vector']):3d}")

In [85]:
lista_reg

[]

In [28]:
for una_ejecucion in lista_execution_name:
    df2 =obtener_datos_ejecucion(una_ejecucion, engine,estandarizar=True)
    df2_filtrado=df2[ df2['label']=='correct']
    df2_medias_por_metrica = df2_filtrado.groupby('metric_name')[['metric_value']].mean()
    break

    
    
